In [1]:
import pandas as pd

In [2]:
# Load dataset
df = pd.read_csv("hr_attrition_dirty.csv")

In [4]:
df.head()

,EmployeeID,EmployeeName,Age,Gender,Department,JobRole,EducationField,Education,MonthlyIncome,HourlyRate,...,BusinessTravel,MaritalStatus,DistanceFromHome,OverTime,PercentSalaryHike,TrainingTimesLastYear,StockOptionLevel,JobInvolvement,Attrition,JoiningDate
0,EMP7367,Daniel Garcia,NaN,Female,IT,NaN,Technical Degree,5,18140,90,...,Non-Travel,Married,11,NaN,23,2,3,1,No,19-Nov-10
1,EMP6604,Daniel Moore,19,Male,R & D,Operations Lead,Other,2,3643,30,...,Non-Travel,divorced,30,NaN,18,4,0,3,no,13-07-2019
2,EMP4384,Sandra Smith,43,Male,Research & Development,Operations Lead,Life Sciences,2,2507,58,...,Travel_Rarely,Married,10,YES,15,6,0,4,Yes,NaN
3,EMP5511,Michael Harris,27,Male,R & D,Marketing Specialist,Marketing,4,4526,91,...,Non-Travel,Married,17,No,16,1,2,1,No,11/24/2018
4,EMP5299,Mark Davis,22,Male,Sales,Laboratory Technician,Medical,5,1637,99,...,Travel_Frequently,Divorced,26,No,23,1,3,3,No,NaN


In [9]:
df.columns

Index(['EmployeeID', 'EmployeeName', 'Age', 'Gender', 'Department', 'JobRole',
       'EducationField', 'Education', 'MonthlyIncome', 'HourlyRate',
       'DailyRate', 'YearsAtCompany', 'YearsInCurrentRole',
       'YearsWithCurrManager', 'TotalWorkingYears', 'NumCompaniesWorked',
       'JobSatisfaction', 'EnvironmentSatisfaction', 'WorkLifeBalance',
       'PerformanceRating', 'RelationshipSatisfaction', 'BusinessTravel',
       'MaritalStatus', 'DistanceFromHome', 'OverTime', 'PercentSalaryHike',
       'TrainingTimesLastYear', 'StockOptionLevel', 'JobInvolvement',
       'Attrition', 'JoiningDate'],
      dtype='str')

In [10]:
df["OverTime"] = (
    df["OverTime"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        "y": "Yes",
        "yes": "Yes",
        "1": "Yes",
        "n": "No",
        "no": "No",
        "0": "No"
    })
)

df["OverTime"] = df["OverTime"].fillna(
    df["OverTime"].mode()[0]
)

In [12]:
# Inconsistent Categories (Attrition)
# yes/YES/Y/1/Yes
# ==================================================
df["Attrition"] = (
    df["Attrition"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        "y": "Yes",
        "yes": "Yes",
        "1": "Yes",
        "n": "No",
        "no": "No",
        "0": "No"
    })
)

In [14]:
# Wrong Data Types
# Age = "twenty five", "?"
# ==================================================
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")

df["Age"] = df["Age"].fillna(df["Age"].median())

In [15]:
# Impossible Outliers (Age)
# Age = -5, 150, 999
# ==================================================
df.loc[(df["Age"] < 18) | (df["Age"] > 65), "Age"] = pd.NA

df["Age"] = df["Age"].fillna(df["Age"].median())

In [16]:
# Currency Symbols
# $5,000
# ==================================================
df["MonthlyIncome"] = (
    df["MonthlyIncome"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
)

df["MonthlyIncome"] = pd.to_numeric(
    df["MonthlyIncome"],
    errors="coerce"
)

In [17]:
# Extra Whitespace
# "  John Smith   "
# ==================================================
df["EmployeeName"] = df["EmployeeName"].str.strip()

In [19]:
# Mixed Date Formats
# 01/15/2022
# 15-01-2022
# 15 Jan 2022
# ==================================================
df["JoiningDate"] = pd.to_datetime(
    df["JoiningDate"],
    errors="coerce",
    dayfirst=True
)

In [21]:
#  Special Characters
# John Smith!!
# ==================================================
df["EmployeeName"] = (
    df["EmployeeName"]
    .str.replace(r"[^A-Za-z\s]", "", regex=True)
)

In [24]:
df = df.dropna(subset=["JoiningDate"])

In [25]:
df["JoiningDate"] = df["JoiningDate"].fillna(
    df["JoiningDate"].mode()[0]
)

In [26]:
median_date = df["JoiningDate"].dropna().median()

df["JoiningDate"] = df["JoiningDate"].fillna(median_date)

In [27]:
df["JoiningDate"] = pd.to_datetime(
    df["JoiningDate"],
    errors="coerce"
)

In [28]:
df["JoiningDate_Clean"] = pd.to_datetime(
    df["JoiningDate"],
    errors="coerce",
    dayfirst=True
)

df[df["JoiningDate_Clean"].isna()][["JoiningDate"]]

,JoiningDate


In [29]:
df["JoiningDate"] = pd.to_datetime(
    df["JoiningDate"],
    errors="coerce",
    dayfirst=True
)

df["JoiningDate"] = df["JoiningDate"].fillna(
    df["JoiningDate"].median()
)

In [30]:
# Clean income
df["MonthlyIncome"] = (
    df["MonthlyIncome"]
      .astype(str)
      .str.replace(r"[$₹,]", "", regex=True)
)

df["MonthlyIncome"] = pd.to_numeric(
    df["MonthlyIncome"],
    errors="coerce"
)

# Fill missing values using JobRole median
df["MonthlyIncome"] = df["MonthlyIncome"].fillna(
    df.groupby("JobRole")["MonthlyIncome"]
      .transform("median")
)

In [31]:
df.isnull().sum()

EmployeeID                    0
EmployeeName                  0
Age                           0
Gender                        0
Department                  536
JobRole                     458
EducationField                0
Education                     0
MonthlyIncome                46
HourlyRate                    0
DailyRate                     0
YearsAtCompany              337
YearsInCurrentRole            0
YearsWithCurrManager          0
TotalWorkingYears           621
NumCompaniesWorked            0
JobSatisfaction             515
EnvironmentSatisfaction       0
WorkLifeBalance             500
PerformanceRating           320
RelationshipSatisfaction      0
BusinessTravel                0
MaritalStatus                 0
DistanceFromHome              0
OverTime                      0
PercentSalaryHike             0
TrainingTimesLastYear         0
StockOptionLevel              0
JobInvolvement                0
Attrition                     0
JoiningDate                   0
JoiningD

In [32]:
df = df.dropna()

In [33]:
df.isnull().sum()

EmployeeID                  0
EmployeeName                0
Age                         0
Gender                      0
Department                  0
JobRole                     0
EducationField              0
Education                   0
MonthlyIncome               0
HourlyRate                  0
DailyRate                   0
YearsAtCompany              0
YearsInCurrentRole          0
YearsWithCurrManager        0
TotalWorkingYears           0
NumCompaniesWorked          0
JobSatisfaction             0
EnvironmentSatisfaction     0
WorkLifeBalance             0
PerformanceRating           0
RelationshipSatisfaction    0
BusinessTravel              0
MaritalStatus               0
DistanceFromHome            0
OverTime                    0
PercentSalaryHike           0
TrainingTimesLastYear       0
StockOptionLevel            0
JobInvolvement              0
Attrition                   0
JoiningDate                 0
JoiningDate_Clean           0
dtype: int64

In [34]:
# Final Check
# ==================================================
print(df.info())
print(df.isnull().sum())

<class 'pandas.DataFrame'>
Index: 2199 entries, 1 to 8297
Data columns (total 32 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   EmployeeID                2199 non-null   str           
 1   EmployeeName              2199 non-null   str           
 2   Age                       2199 non-null   float64       
 3   Gender                    2199 non-null   str           
 4   Department                2199 non-null   str           
 5   JobRole                   2199 non-null   str           
 6   EducationField            2199 non-null   str           
 7   Education                 2199 non-null   int64         
 8   MonthlyIncome             2199 non-null   float64       
 9   HourlyRate                2199 non-null   int64         
 10  DailyRate                 2199 non-null   int64         
 11  YearsAtCompany            2199 non-null   float64       
 12  YearsInCurrentRole        2199 non-n

In [36]:
# Inconsistent Categories (Gender)
# male/MALE/M/m/fEmale
# ==================================================
df["Gender"] = (
    df["Gender"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        "m": "Male",
        "male": "Male",
        "f": "Female",
        "female": "Female"
    })
)

In [39]:
# Save cleaned dataset
df.to_csv("HR_Attrition_Cleaned.csv", index=False)